In [ ]:
#
#  homework 1 agentic rag
# 

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

i = 0

for file in files:
    doc = file.parse()
    documents.append(doc)
    print(f"{doc['filename']}: {doc.get('title', 'No title')}")
    i = i + 1
    
print(f"\nNumber of lesson pages in data set:", i,"\n")

01-agentic-rag/lessons/01-intro.md: No title
01-agentic-rag/lessons/02-environment.md: No title
01-agentic-rag/lessons/03-rag.md: No title
01-agentic-rag/lessons/04-dataset.md: No title
01-agentic-rag/lessons/05-search.md: No title
01-agentic-rag/lessons/06-building-prompt.md: No title
01-agentic-rag/lessons/07-llm.md: No title
01-agentic-rag/lessons/08-rag-helper.md: No title
01-agentic-rag/lessons/09-data-ingestion.md: No title
01-agentic-rag/lessons/10-rag-next-steps.md: No title
01-agentic-rag/lessons/11-agents-intro.md: No title
01-agentic-rag/lessons/12-rag-revision.md: No title
01-agentic-rag/lessons/13-function-calling.md: No title
01-agentic-rag/lessons/14-agentic-loop.md: No title
01-agentic-rag/lessons/15-frameworks.md: No title
01-agentic-rag/lessons/16-other-frameworks.md: No title
02-vector-search/lessons/01-intro.md: No title
02-vector-search/lessons/02-embeddings.md: No title
02-vector-search/lessons/03-embeddings-dataset.md: No title
02-vector-search/lessons/04-vector-

In [3]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [4]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results = 5
)

# search_results
[doc["filename"] for doc in search_results]

['01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/11-agents-intro.md',
 '01-agentic-rag/lessons/16-other-frameworks.md']

In [5]:
from rag_helper import RAGBase
from openai import OpenAI
import os

BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:1234/v1")
API_KEY = os.getenv("OPENAI_API_KEY", "not-needed")

openai_client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [6]:
print()
print("Ask question: How does the agentic loop keep calling the model until it stops?\n")
answer, input_tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print("Answer is\n")
print(answer)
print()
print(f"Number of input tokens:", input_tokens, "\n")


Ask question

Answer is

The agentic loop keeps calling the model until it stops by using a **`while True` loop** with an exit condition based on whether the model requests further tool calls (`function_call`). Here’s how it works:

### Key Mechanism:
1. **Loop Continuation**:
   - The loop runs indefinitely (`while True`) until explicitly broken.
   - Each iteration processes the current conversation state, including any function calls or messages from the previous turn.

2. **Exit Condition**:
   - After each model response, the code checks if there are any `function_call` entries in its output using a flag (`has_function_calls`).
   - If no `function_call` is found, the loop breaks and returns the final answer.
     ```python
     while True:
         # Process model response...
         for item in response.output:
             if item.type == "function_call":
                 has_function_calls = True  # Flag set to true
             elif item.type == "message":
                 

In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


In [10]:
index_chunks = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunks.fit(chunks)

In [11]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results_chunks = index_chunks.search(
    question,
    num_results = 5
)

# search_results
[doc["filename"] for doc in search_results_chunks]

['01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/15-frameworks.md']

In [12]:
assistant_chunks = RAGBase(
    index=index_chunks,
    llm_client=openai_client,
)

In [13]:
print()
print("Ask question: How does the agentic loop keep calling the model until it stops?\n")
answer, input_tokens = assistant_chunks.rag("How does the agentic loop keep calling the model until it stops?")
print("Answer is\n")
print(answer)
print()
print(f"Number of input tokens:", input_tokens, "\n")


Ask question: How does the agentic loop keep calling the model until it stops?

Answer is

The **agentic loop** in this implementation keeps calling the model until it stops generating function calls (tool invocations). Here’s how it works:

### How It Stops:
1. **Core Logic**:
   - The loop runs indefinitely (`while True`) until a condition breaks.
   - Each iteration checks if the model returns any `function_call` in its response.

2. **Exit Condition**:
   - If no function calls are detected (`has_function_calls == False`), the loop exits with:
     ```python
     if has_function_calls == False:
         break
     ```
   - This happens when the model provides a final answer (e.g., after refining search results or confirming an action).

3. **Mechanism**:
   - The model iteratively asks for tools (e.g., searches) until it decides to stop.
   - Each tool call updates `messages` and triggers another round of reasoning, propagating the result back into the conversation.

### Example F